In [2]:
import pandas as pd
import numpy as np
from pathlib import Path
import joblib

In [3]:
from data_cleaning import clean_df
from feature_engineering import feature_engineering
from sklearn_utilities import group_shuffle_split, scale_dataset 
from utilities import sort_values, num_aircrafts, initialise_features_and_target, initialise_df, integrate_weather_data
from rnn_sequences import save_sequences, save_test_sequences
from fetch_weather_data import get_weather_data

In [4]:
df_test = initialise_df('../data/opensky_test.csv')

In [5]:
df_test['raw_latitude'] = df_test['latitude']
df_test['raw_longitude'] = df_test['longitude']

In [6]:
df_test.isnull().sum()

icao24                0
callsign          11951
origin_country     4264
timestamp          5667
longitude          5667
latitude           5667
baro_altitude     53484
on_ground             0
velocity            150
true_track            0
vertical_rate     50814
geo_altitude      57860
category              0
raw_latitude       5667
raw_longitude      5667
dtype: int64

In [7]:
df_test = sort_values(df_test)
df_clean = clean_df(df_test)

In [8]:
df_clean['timestamp'] = pd.to_datetime(df_clean['timestamp'], unit='s')
    
center_lat = df_clean['latitude'].mean()
center_lon = df_clean['longitude'].mean()
start_date = df_clean['timestamp'].min().strftime('%Y-%m-%d')
end_date = (df_clean['timestamp'].max() + pd.Timedelta(days=1)).strftime('%Y-%m-%d')

df_weather = get_weather_data(center_lat, center_lon,
                              start_date, end_date)
df_clean = integrate_weather_data(df_clean, df_weather)

Fetching weather and thermodynamic data for given parameters..
Trying Archive API...


c:\Users\yashdeshpande\Desktop\serious_projects\Serenair-Flight-Trajectory-Detection\preprocessing\fetch_weather_data.py:61: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['wind_speed'] = df['wind_speed_high'].fillna(df['wind_speed_low'])
c:\Users\yashdeshpande\Desktop\serious_projects\Serenair-Flight-Trajectory-Detection\preprocessing\fetch_weather_data.py:62: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['wind_dir'] = df['wind_dir_high'].fillna(df['wind_dir_low'])
c:\Users\yashdeshpande\Desktop\serious_projects\Serenair-Flight-Trajectory-Detection

Successfully retrieved 48 hours of thermodynamic weather data!
Fusing both the datasets


In [9]:
df_clean.head()

,timestamp,icao24,origin_country,longitude,latitude,baro_altitude,on_ground,velocity,true_track,vertical_rate,category,raw_latitude,raw_longitude,wind_speed,wind_dir,temperature
9,2026-06-16 08:57:20,344043,Spain,2.9244,39.5269,6797.04,True,0.0,165.94,0.0,0.0,39.5269,2.9244,4.865556,251.222222,21.295556
10,2026-06-16 08:57:20,344043,Spain,2.9244,39.5269,6797.04,True,0.0,165.94,0.0,0.0,39.5269,2.9244,4.865556,251.222222,21.295556
11,2026-06-16 08:57:20,344043,Spain,2.9244,39.5269,6797.04,True,0.0,165.94,0.0,0.0,39.5269,2.9244,4.865556,251.222222,21.295556
12,2026-06-16 08:57:20,344043,Spain,2.9244,39.5269,6797.04,True,0.0,165.94,0.0,0.0,39.5269,2.9244,4.865556,251.222222,21.295556
13,2026-06-16 08:57:20,344043,Spain,2.9244,39.5269,6797.04,True,0.0,165.94,0.0,0.0,39.5269,2.9244,4.865556,251.222222,21.295556


In [10]:
df_test = feature_engineering(df_clean)

In [13]:
df_test.isnull().sum()

timestamp              0
icao24                 0
origin_country      4197
longitude              0
latitude               0
baro_altitude          0
on_ground              0
velocity               0
true_track             0
vertical_rate          0
raw_latitude           0
raw_longitude          0
wind_speed             0
wind_dir               0
temperature            0
delta_time             0
track_sin              0
track_cos              0
hour_sin               0
hour_cos               0
prev_velocity          0
acceleration           0
prev_track             0
turn_rate              0
climb_phase            0
delta_latitude         0
delta_longitude        0
wind_dir_radians       0
cat_Heavy              0
cat_Large              0
cat_Light              0
cat_Medium             0
cat_Small              0
cat_Super              0
temp_kelvin            0
is_emergency           0
dtype: int64

In [18]:
save_dir = Path('../data/rnn_data/')
scaler_path = save_dir / 'feature_scaler.joblib'
target_path = save_dir / 'target_scaler.joblib'

In [21]:
feature_scaler = joblib.load(scaler_path)
feature_scaler

,copy,True
,with_mean,True
,with_std,True


In [22]:
target_scaler = joblib.load(target_path)
target_scaler

,with_centering,True
,with_scaling,True
,quantile_range,"(25.0, ...)"
,copy,True
,unit_variance,False


In [24]:
features, target = initialise_features_and_target()

In [ ]:
df_test[features] = feature_scaler.transform(df_test[features])

,velocity,vertical_rate,baro_altitude,delta_time,track_sin,track_cos,acceleration,turn_rate,climb_phase,hour_sin,hour_cos,euclidean_speed,bearing
9107,-1.776438,0.713901,-1.521094,-2.133871e+07,0.541046,-1.864545,0.001037,0.01682,3.088226,0.0,-1.0,-0.131230,-0.001038
21118,-1.776223,0.713901,-1.521087,-5.398952e+06,0.542703,-1.864080,0.001037,0.01682,3.088226,0.0,-1.0,-0.099648,1.056454
33140,-1.773896,0.188510,-1.521079,1.213478e+07,0.543034,-1.863987,0.001037,0.01682,3.088226,0.0,-1.0,-0.058463,1.056121
45160,-1.773896,0.188510,-1.521079,-2.133871e+07,0.543034,-1.863987,0.001037,0.01682,3.088226,0.0,-1.0,-0.131230,-0.001038
57191,-1.771701,0.574589,-1.521073,1.372876e+07,1.138695,-1.578990,0.001037,0.01682,3.088226,0.0,-1.0,-0.042400,1.019277
...,...,...,...,...,...,...,...,...,...,...,...,...,...
60024,-1.756275,0.219415,-1.520685,1.213478e+07,-1.847481,-0.513504,0.001037,0.01682,3.088226,0.0,-1.0,0.078178,-0.678167
72058,-1.756275,0.219415,-1.520683,-3.804976e+06,-1.847481,-0.513504,0.001037,0.01682,3.088226,0.0,-1.0,-0.021413,-0.679244
84100,-1.756189,0.188510,-1.520679,1.054081e+07,-1.846753,-0.515877,0.001037,0.01682,3.088226,0.0,-1.0,0.065411,-0.678803
96129,-1.756189,0.172819,-1.520678,-3.804976e+06,-1.846753,-0.515877,0.001037,0.01682,3.088226,0.0,-1.0,-0.026241,-0.679776


In [29]:
df_test[target] = target_scaler.transform(df_test[target])

In [30]:
df_test[target]

,delta_latitude,delta_longitude
9107,0.000000,0.000000
21118,-0.585526,0.143564
33140,-1.348684,0.331683
45160,0.000000,0.000000
57191,-1.585526,0.524752
...,...,...
60024,-0.967105,-2.985149
72058,-0.513158,-1.564356
84100,-0.914474,-2.801980
96129,-0.493421,-1.495050


In [33]:
save_test_sequences(df_test, features, target, window_size=10)

Extracting Flight Sequences: 100%|██████████| 12269/12269 [00:08<00:00, 1397.69it/s]


ValueError: need at least one array to concatenate

In [36]:
df_test[features] = feature_scaler.transform(df_test[features])
df_test[target] = target_scaler.transform(df_test[target])
    
# --- DEBUGGING BLOCK ---
print(f"Total rows remaining in df_test: {len(df_test)}")
if len(df_test) > 0:
    flight_lengths = df_test.groupby('icao24').size()
    print(f"Total unique flights: {len(flight_lengths)}")
    print(f"Max flight length: {flight_lengths.max()} rows")
    valid_flights = (flight_lengths > 10).sum()
    print(f"Flights longer than 10 rows: {valid_flights}")
# -----------------------

save_test_sequences(df_test, features, target, window_size=10)

Total rows remaining in df_test: 107037
Total unique flights: 12269
Max flight length: 9 rows
Flights longer than 10 rows: 0


Extracting Flight Sequences: 100%|██████████| 12269/12269 [00:08<00:00, 1421.99it/s]


ValueError: need at least one array to concatenate